# Do two components preserve useful well-log information?
Liam Mora — INDE 577

## Investigation contract
This supporting study asks how much of the standardized GR, RHOB and NPHI variation
can be represented in two dimensions, and what compression costs for sonic-log
prediction in unseen wells. PCA is a representation tool, not a fourth core custom
algorithm. A compact visualization can help an analyst inspect measurement
relationships; high retained variance alone does not establish predictive adequacy.

We compare one, two and all three components without selecting a new production
policy. The three original predictors remain the declared model inputs. All
evidence is development analysis; the 20 final-test wells remain excluded.

## Data provenance and population
Data come from the [FORCE 2020 repository](https://github.com/bolgebrygg/Force-2020-Machine-Learning-competition)
and [archived release](https://zenodo.org/records/4351156), under NLOD 2.0.
The archive was verified November 19, 2026; its checksum is in
`configs/reserved_wells.json`. See `docs/FORCE_DATA_CHECK.md` for acquisition.
Reruns require the ignored local archive `data/local/force-train.zip`.

We deliberately reuse the regression population for an identical-sample comparison:
14,500 training observations from 58 wells and 135,154 validation observations
from 20 other wells. These require observed DTC and complete GR (API), RHOB
(g/cm³) and NPHI (porosity fraction). Of 706,823 training rows, 413,532 qualify
before sampling; of 240,136 validation rows, 135,154 qualify. Missingness causes
selection bias: this is not a description of all available logs.

The adapter rejects invalid schema, duplicate well/depth pairs and sentinel values.
Well identity and depth define dependence/grouping, not model predictors.
Training samples are capped at 250 rows per well with seed 577. Standardization
and PCA fit training predictors only. DTC enters downstream regression and plot
coloring, never the fitting of PCA axes. Labels do not define the projection.

## Mathematical model
Let Z contain training-standardized predictors, using training means and standard
deviations. Center Z by its training column means and compute its singular value
decomposition Zc = U S Vᵀ. The first k rows of Vᵀ define orthonormal axes; scores
are Zc V_k. Explained variance is S_j²/(n−1), divided by total training variance.
Reconstruction is scores V_kᵀ plus the fitted PCA mean.

Standardization prevents units alone from determining the axes. We use full SVD,
no whitening and k = 1, 2, 3. PCA minimizes squared reconstruction error in
standardized predictor space, not DTC prediction error. Signs of axes are arbitrary;
nearly equal eigenvalues can make individual axes unstable. With all three axes,
the rotation loses no information and should preserve unregularized linear predictions.

## Implementation and verification
The professor's template `experiments/unsupervised.py::run_pca_example` uses
StandardScaler followed by scikit-learn PCA. We follow that supporting approach
in `dsm_project.experiments.force_pca`, adapting fitting to training wells only.
The available Lecture 9 landscape notebook supplies conceptual context; no
dedicated PCA implementation notebook was found in the reviewed course snapshot.
PCA and scaling are library implementations; downstream regression uses the
project's custom LectureLinearRegressor.

Tests verify training-only scaling, unchanged fitted axes after validation
transforms, independence of PCA axes from target labels, orthonormality, full
reconstruction and agreement with uncompressed least squares. The next cell runs
the experiment and verifies well separation.

In [ ]:
from pathlib import Path
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "configs/force.toml").is_file())
os.chdir(ROOT)
from dsm_project.data.force import FEATURES
from dsm_project.experiments.force_pca import run_experiment

result = run_experiment()
report = result["report"]
training, validation = result["training"], result["validation"]
reserved = set(json.loads((ROOT / "configs/reserved_wells.json").read_text())["reserved_test_wells"])
assert not set(training.WELL) & set(validation.WELL)
assert not (set(training.WELL) | set(validation.WELL)) & reserved
assert report["full_rotation_max_prediction_gap"] < 1e-4
print("Archive SHA256:", report["source_sha256"])
print("Training sample SHA256:", report["sample_ids_sha256"])
display(training.loc[:, list(FEATURES)].describe().T)
display(training.loc[:, list(FEATURES)].corr())

## Exploration: redundancy and variance
Training correlations show linear associations before rotation; they cannot prove
that a variable is dispensable for prediction. The next plot shows retained
training variance and the axis coefficients, also called component weights here.
These coefficients are not feature-target effects or causal importance.

In [ ]:
ratios = np.array(report["explained_variance_ratio"])
weights = np.array(report["components"])
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
axes[0].bar([1, 2, 3], ratios, label="Individual component")
axes[0].plot([1, 2, 3], ratios.cumsum(), marker="o", color="#c46b22", label="Cumulative")
axes[0].set(xlabel="Component", ylabel="Training variance fraction", xticks=[1, 2, 3], ylim=(0, 1.05))
axes[0].legend()
im = axes[1].imshow(weights, cmap="coolwarm", vmin=-1, vmax=1)
axes[1].set(xticks=[0, 1, 2], xticklabels=FEATURES, yticks=[0, 1, 2],
            yticklabels=["PC1", "PC2", "PC3"], title="Orthonormal axis coefficients")
for (i, j), value in np.ndenumerate(weights):
    axes[1].text(j, i, f"{value:.2f}", ha="center", va="center")
fig.colorbar(im, ax=axes[1], label="Coefficient")
plt.show()
print(f"Two components retain {ratios[:2].sum():.1%} of training variance.")

A large retained fraction supports compact visualization, not automatic feature
removal. The omitted direction may still carry target-relevant signal. Signs can
reverse without changing the represented subspace.

## Validation projection and reconstruction
The following views use one reproducible plotting sample of up to 200 observations
per validation well. Sampling is for display only; numerical scores use all eligible
validation observations. Coordinates use the training-fitted scaler and PCA.
DTC colors are an external audit, not information used to fit the representation.

In [ ]:
from dsm_project.experiments.force_linear import sample_training_wells
shown = sample_training_wells(validation, rows_per_well=200, seed=577)
pipeline = result["models"][2]
scaler, pca = pipeline.steps[0][1], pipeline.steps[1][1]
scaled = scaler.transform(shown.loc[:, FEATURES])
scores = pca.transform(scaled)
reconstruction = np.mean((scaled - pca.inverse_transform(scores))**2, axis=1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
for ax, color, title, label in [
    (axes[0], shown.DTC, "Sonic values across the projection", "Observed DTC (µs/ft)"),
    (axes[1], reconstruction, "Information outside the two-axis plane", "Standardized reconstruction MSE"),
]:
    scatter = ax.scatter(scores[:, 0], scores[:, 1], c=color, s=8, alpha=.65, cmap="viridis")
    ax.set(xlabel="PC1 score (standardized units)", ylabel="PC2 score (standardized units)", title=title)
    fig.colorbar(scatter, ax=ax, label=label)
plt.show()

Color gradients and overlapping points reveal associations and loss of information,
not distinct geological clusters. Reconstruction error measures distance from the
retained plane; it is not a calibrated anomaly score, a geological diagnosis, or
an uncertainty estimate for DTC. Points outside a training-dense region deserve
review, but no rejection threshold is selected from this plot.

## Failure analysis and prediction comparison
Each dimensionality feeds the same custom linear estimator trained on the same
sample. Compare both prediction error and standardized reconstruction error.
Reconstruction error and DTC error have different units and objectives.

In [ ]:
comparison = pd.DataFrame(report["comparisons"])
display(comparison.round(6))
print("Uncompressed linear baseline:", report["uncompressed_linear"])
print("Full rotation maximum prediction difference:", report["full_rotation_max_prediction_gap"])
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), constrained_layout=True)
axes[0].plot(comparison.components, comparison.validation_mae, marker="o", label="PCA + custom linear")
axes[0].axhline(report["uncompressed_linear"]["mae"], color="gray", linestyle="--", label="Uncompressed linear")
axes[0].set(xlabel="Retained components", ylabel="Validation MAE (µs/ft)", xticks=[1, 2, 3])
axes[0].legend()
axes[1].plot(comparison.components, comparison.validation_standardized_reconstruction_mse, marker="o")
axes[1].set(xlabel="Retained components", ylabel="Validation standardized reconstruction MSE", xticks=[1, 2, 3])
plt.show()
per_well = result["per_well"].pivot(index="well", columns="components", values="mae")
per_well["two_minus_three"] = per_well[2] - per_well[3]
display(per_well.sort_values("two_minus_three").round(3))

Two components retain 91.0% of training variance. Validation MAE is
10.27 µs/ft versus 10.52 for the uncompressed linear model (about 2.4% lower);
RMSE falls from 14.94 to 14.46. One component has MAE 10.83 and performs worse.
This modest two-component improvement is an observed development result, not
proof of a stable gain. Removing a direction can reduce noise as well as discard
signal. Repeated well-level evaluation would be needed before adopting compression.

The full three-component result is an implementation check: an orthogonal rotation
with an intercept does not change the unregularized linear hypothesis class.
Compression removes directions and can change prediction error even when retained
variance is high. Per-well errors show how that cost varies geographically; the
pooled score weights wells with more observations more heavily.

This is one development split with dependent depth observations, not an uncertainty
interval. PCA is sensitive to scaling, finite extremes and changes in covariance.
No robust scaling, clipping, component count or anomaly threshold is selected here.
The regression complete-case population further restricts interpretation.

## Conclusion and production decision
PCA provides an inspectable two-dimensional view of the measurement relationships.
It does not establish that compressing three already-manageable predictors improves
our product. Retain the original three inputs for the supervised comparisons;
use PCA as supporting representation evidence only. No new model is promoted and
no geological or causal claims follow from the projection.

Next, complete rock-type failure analysis and repeated whole-well validation.
Freeze modeling and selection decisions before scoring the 20 reserved wells.

## Reproduce
Run all cells using DSM Project with the audited archive available.
Package runner: `uv run --locked python -m dsm_project.experiments.force_pca`.
Tests: `uv run --locked pytest tests/test_force_pca.py tests/test_pca_notebook.py -q`.
The full notebook execution test skips without local field data; structural and
synthetic tests remain active in CI. Committed outputs are cleared. The executed
review copy remains local at `artifacts/force_pca/04-pca-representation.executed.ipynb`.
Candidate metrics and per-well tables are saved in the same ignored directory.